# BFND: Supervised run guide (00)

**Ei notebook-er kaj:** shob dhap ek jaygay, ekta ekta kore. Prottek phase-er shesh-e ⛔ **STOP** lekha thakbe। Tokhon output-ta chat-e paste korben, ami dekhe bolbo porer dhap-ta chalabe kina.

**Rules**
1. Ekbar ekta phase cholbe। Parallel-e na।
2. Kono cell error dile, error-ta puro paste korben। Ami thik kore dibo, tarpor abar chalabon।
3. Human check sheet (⛔ Phase 1 ar Phase 2) bhoroshajogo na hole porer result-er upor bharosha korben na।
4. Disconnect hole: shei phase-er cell abar chalan। Cache thakle kaj resume hobe।

| Phase | Ki hobe | Lage | Ke kore |
|---|---|---|---|
| 0 | Setup: GPU, Drive, code unpack | 5 min | apni |
| 1 | Cultural silver label (LLM) + 40-row human check | model download + kichhu ghonta | script + apni (40 row) |
| 2 | Wikipedia evidence + 40-row relevance check | kichhu ghonta | script + apni (40 row) |
| 3 | Feature refresh (BanglaBERT, CLIP, cultural, evidence) | kichhu minute | script |
| 4 | Backbone feature (mBERT, XLM-R, ViT, Swin) | kichhu minute | script |
| 5 | Selection grid: 35 configuration x 3 seed = 105 run | kichhu ghonta | script |
| 6 | (optional) Ager E0–E6 ablation (04) | kichhu ghonta | script |
| 7 | Winner + cultural/evidence stage | **ekhono baki** | — |

**Model:** cultural ar query-er jonno free local model (`Qwen/Qwen2.5-3B-Instruct`), GPU-te chole। Config: `configs/config.py`।


## Phase 0: Setup

GPU T4 thakte hobe (Runtime > Change runtime type > T4 GPU).

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, zipfile
from pathlib import Path

ZIP = Path("/content/drive/MyDrive/Bangla-Multimodal-FakeNews.zip")
CODE_DIR = Path("/content/Bangla-Multimodal-FakeNews")
assert ZIP.exists(), "Code zip MyDrive-e nei. Latest zip upload korun."
shutil.rmtree(CODE_DIR, ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content")
os.chdir(CODE_DIR)
cfg = Path("configs/config.py").read_text(encoding="utf-8")
assert "TEXT_BACKBONES" in cfg and "LLM_BACKEND" in cfg, "Purono code. Latest zip upload korun."
print("cwd:", os.getcwd(), "| code version OK")

In [ ]:
!pip install -q "accelerate>=0.26" "transformers>=4.46" "sentence-transformers>=3.0" "rank-bm25>=0.2.2" joblib openpyxl requests

In [ ]:
from configs.config import BASE, RAW_DIR, PROCESSED_DIR, FEATURE_DIR, LLM_BACKEND, LOCAL_LLM_MODEL
print("BASE:", BASE)
print("LLM backend:", LLM_BACKEND, "|", LOCAL_LLM_MODEL)
for p in [RAW_DIR, PROCESSED_DIR / "unified_split.csv", FEATURE_DIR / "meta.csv", FEATURE_DIR / "bert.npy", FEATURE_DIR / "clip_img.npy"]:
    print(("OK   " if p.exists() else "MISS ") + str(p))

⛔ **STOP 0:** upor-er cell-er output paste korun। `MISS` thakle ami bolbo kon phase-e ki bujhte hobe।

Jodi `meta.csv`, `bert.npy`, `clip_img.npy` MISS hoy, tahole age notebook 02 chalate hobe (feature build)।

## Phase 1: Cultural silver label

**Ki hobe:** LLM (Qwen) prottek post-er 4-ti cultural feature (sarcasm, humor, local_reference, historical_reference) 0/1 diye dibe। Label-er class info pathano hoy na।
**Prothom run:** silver label + ekta 40-row check sheet toiri hobe।

In [ ]:
!python scripts/05_build_cultural_silver.py

### ⛔ 1A: Human check (apnar kaj, 40 row)

1. Drive-e khunjun: `MyDrive/BFND-data/annotations/cultural_human_check.csv`
2. Google Sheets-e khulun (ba Colab-e pandas diye)।
3. `sarcasm`, `humor`, `local_reference`, `historical_reference` column-e 0 ba 1 din। `text` pore padhe decide korben।
4. Row-er order na badlaun। Sob 40 row bharun (na jante parle 0 din, tarpor ami bolbo)।
5. Save korun, tarpor porer cell chalan।


In [ ]:
!python scripts/05_build_cultural_silver.py

In [ ]:
import json
import pandas as pd
from configs.config import RESULTS_DIR, CULTURAL_SILVER_CSV, CULTURAL_CHECK_CSV

if not CULTURAL_SILVER_CSV.exists():
    print("MISSING:", CULTURAL_SILVER_CSV)
    print("Silver label file ekhono toiri hoyni. Ei karon-er ekta:")
    print("  (a) Upore 'Phase 1' er script cell (05) shesh hoyni: cell-ta ekhono chalchhe ba thamiyeche, shesh na hole abar chalan (cache thakle resume hobe)")
    print("  (b) 05 cell-e error hoyeche: oi cell-er output (error) paste korun")
    print("Ei cell-ta 05 shesh holeo abar chalan।")
    raise SystemExit
silver = pd.read_csv(CULTURAL_SILVER_CSV)
print("Silver rows:", len(silver))
print("Positive rate per feature (LLM):")
print(silver.drop(columns="sample_id").mean().round(3).to_string())

agr_path = RESULTS_DIR / "cultural_agreement.json"
if not agr_path.exists():
    print("\nAgreement nei. Human check sheet ta bharun, tarpor is cell abar chalan.")
else:
    report = json.loads(agr_path.read_text(encoding="utf-8"))
    print("\nAgreement with human labels (proposed thresholds: PASS >= 0.6, REVIEW 0.4-0.6, FAIL < 0.4):")
    for name, r in report.items():
        k = r.get("kappa")
        if k is None:
            status = "NO DATA (human has not filled this feature)"
        elif k >= 0.6:
            status = "PASS"
        elif k >= 0.4:
            status = "REVIEW"
        else:
            status = "FAIL: do not use this feature"
        print(f"  {name:<22} n={r.get('n')}  agreement={r.get('agreement')}  kappa={k}  -> {status}")


⛔ **STOP 1:** Silver rate ar agreement table paste korun।

Ami dekhbo: kono feature FAIL hole seta E4-te bad dibo। Thresholds proposal, apni bolle change korbo।

## Phase 2: Wikipedia evidence

**Ki hobe:** LLM prottek post-er theke Bangla search query banabe, Wikipedia theke article-er intro niye passage-e bhag kora hobe। Label kono jaygay byabohar hoy na।
**Prothom run:** evidence.csv + coverage report + 40-row relevance sheet।

In [ ]:
!python scripts/06_build_evidence_wiki.py

### ⛔ 2A: Relevance check (apnar kaj, 40 row)

1. Drive: `MyDrive/BFND-data/annotations/evidence_relevance_check.csv`
2. `text`, `query`, `wiki_title` dekhe `relevant_0_or_1`-e 1 (Wikipedia article ta post-er fact-er sathe sambhab) ba 0 din।
3. Row-er order badlaben na। Save korun।


In [ ]:
import json
import pandas as pd
from configs.config import RESULTS_DIR, EVIDENCE_RELEVANCE_CHECK_CSV

from configs.config import EVIDENCE_CSV
if not (RESULTS_DIR / "evidence_build_report.json").exists():
    print("MISSING: evidence_build_report.json. Upore 'Phase 2' script cell (06) shesh hoyni ba error diyeche.")
    print("Shei cell-er output paste korun; shesh na hole abar chalan।")
    raise SystemExit
rep = json.loads((RESULTS_DIR / "evidence_build_report.json").read_text(encoding="utf-8"))
print(json.dumps(rep, ensure_ascii=False, indent=2))
cov = rep["coverage_share"]
cov_status = "PASS" if cov >= 0.5 else ("REVIEW" if cov >= 0.3 else "WEAK: E5 gain will likely be small")
print(f"\nCoverage {cov:.3f} -> {cov_status}   (proposed: PASS >= 0.5, REVIEW 0.3-0.5)")

check = pd.read_csv(EVIDENCE_RELEVANCE_CHECK_CSV)
filled = check["relevant_0_or_1"].dropna()
if len(filled) == 0:
    print("Relevance nei. Sheet ta bharun, tarpor is cell abar chalan.")
else:
    rate = filled.astype(int).mean()
    rel_status = "PASS" if rate >= 0.6 else ("REVIEW" if rate >= 0.4 else "FAIL")
    print(f"Relevance {rate:.3f} on {len(filled)} rows -> {rel_status}   (proposed: PASS >= 0.6, REVIEW 0.4-0.6)")


⛔ **STOP 2:** Coverage ar relevance output paste korun। Relevance FAIL hole E5 chalanor age ami bolbo।

## Phase 3: Feature refresh

**Ki hobe:** script 02 abar chalabe। Eta BanglaBERT ar CLIP feature abar banabe (kichhu minute), ar cultural.npy ar evidence.npy toiri korbe।
Cultural source: manual file nei, tai `llm_silver` use hobe (`results/cultural_source.json`-e lekha)।

In [ ]:
!python scripts/02_extract_features.py

In [ ]:
import json
import numpy as np
from configs.config import FEATURE_DIR, RESULTS_DIR

for name in ["bert", "clip_img", "clip_txt", "sim", "has_image", "cultural", "evidence"]:
    p = FEATURE_DIR / f"{name}.npy"
    print(f"{name:<10}", np.load(p).shape if p.exists() else "MISSING")
src = RESULTS_DIR / "cultural_source.json"
print("\ncultural source:", json.loads(src.read_text()) if src.exists() else "not built")
guard = RESULTS_DIR / "retrieval_report.json"
print("retrieval report:", json.loads(guard.read_text()) if guard.exists() else "not built")

⛔ **STOP 3:** Shape table ar retrieval report paste korun।

## Phase 4: Backbone features (mBERT, XLM-R, ViT, Swin)

**Ki hobe:** `bert.npy` ar `clip_img.npy` reuse hobe। Baki 4-ti feature notun kore banabe।

In [ ]:
!python scripts/07_extract_backbones.py

⛔ **STOP 4:** Output paste korun (zero-row check `True` hote hobe)।

## Phase 5: Selection grid

**Ki hobe:** 3 text × 3 image × 3 fusion = 27 combination, plus 3 text-only, 3 image-only, 2 control. Prottek 3 seed-e train. Winner validation macro-F1 diye select korbe, test-e kono choice nei।
**Lage:** kichhu ghonta। Ekta run ekta line print korbe।

In [ ]:
!python scripts/08_model_selection.py

In [ ]:
import json
import numpy as np
from configs.config import SELECTION_DIR, LABELS_3
from src.selection import format_table, confusion_text

if not (SELECTION_DIR / "summary.json").exists():
    print("MISSING: results/selection/summary.json. Phase 5 script cell (08) shesh hoyni ba error diyeche; output paste korun।")
    raise SystemExit
summary = json.loads((SELECTION_DIR / "summary.json").read_text(encoding="utf-8"))
cols = [("val_macro_f1", "Val F1"), ("test_accuracy", "Test Acc"), ("test_macro_precision", "Test Prec"),
        ("test_macro_recall", "Test Recall"), ("test_macro_f1", "Test F1")]
for kind, title in [("text", "TEXT ONLY"), ("image", "IMAGE ONLY"), ("control", "CONTROLS"), ("fusion", "FUSION")]:
    names = sorted([n for n, r in summary.items() if r["kind"] == kind], key=lambda n: -summary[n]["val_macro_f1_mean"])
    print("=" * 110); print(title); print(format_table(summary, names, cols)); print()
for name in sorted(summary, key=lambda n: (summary[n]["kind"], -summary[n]["val_macro_f1_mean"])):
    print("=" * 80); print(name)
    print(confusion_text(np.array(summary[name]["test_confusion_mean"]), LABELS_3)); print()
best = json.loads((SELECTION_DIR / "best_config.json").read_text(encoding="utf-8"))
print("WINNERS (validation-only):", best)

⛔ **STOP 5:** Sob table ar winner-er line paste korun। Prothom ami dekhbo grid-er shob run shesh hoyeche kina (105 run)।

Notebook `03_model_selection.ipynb` theke plot-o dekhte parben।

## Phase 6 (optional): Ager E0–E6 ablation

**Ki hobe:** ager script 04-er experiment: E0 (presence), E1f, E2, E3, E4 (+cultural), E5 (+evidence), E6 (decision layer)। Eta grid-er winner ar bhinno base (BanglaBERT + CLIP concat) use kore।
Ei phase-tai E4/E5/E6-er ekta result ditey pare; kintu oita grid-er winner nay।

In [ ]:
!python scripts/04_train_fusion.py

In [ ]:
from configs.config import RESULTS_DIR
print((RESULTS_DIR / "summary.md").read_text(encoding="utf-8"))

⛔ **STOP 6:** `summary.md` paste korun।

## Phase 7: Ki baki ache (ekhono bana hoyni)

1. **Winner + cultural + evidence:** grid-er winner fusion-er upor E4 (cultural) ar E5 (evidence) ar E6 (decision layer)। Ei kaj ekhono code-e nai।
2. **Test set-e ekbar report:** selection shesh holeo test-er final table ar bootstrap confidence interval।
3. **Ager E4/E5/E6 ar grid-er winner:** kon base-e E4-E6 chalabo, tomar decision lagbe।
4. **App integration:** saved head (`checkpoints/selection/`) diye Gradio app-e prediction।
5. **Disconnect-resume:** script 08 ekhon shuru theke abar chalay, mid-grid resume nei।

Ei gulo porer step-e ami build korbo, age Phase 1–6 shesh hok।